## ⚙️ Cellule 1 — Configuration

In [9]:
from google.colab import drive
import os, json

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive déjà monté')

# ─── CHEMINS (identiques aux notebooks YOLO/Faster R-CNN, sauf SAVE_DIR / dossier local) ──
FRACATLAS_ZIP    = '/content/drive/MyDrive/FracAtlas/archive.zip'
FRACATLAS_ROOT   = '/content/FracAtlas_extracted'
IMAGES_SUBDIRS   = ['Fractured', 'Non_fractured']
LABELS_DIR       = os.path.join(FRACATLAS_ROOT, 'Annotations', 'YOLO')
SPLITS_PATH      = '/content/drive/MyDrive/resultats_finaux_fracture/splits_fracatlas.json'
STATS_PATH       = '/content/drive/MyDrive/resultats_finaux_fracture/fracatlas_clahe_stats.json'
SAVE_DIR         = '/content/drive/MyDrive/resultats_finaux_fracture/RTDETR'
YOLO_DATA        = '/content/rtdetr_fracatlas_dataset'  # dossier local dédié (cf. notebooks précédents :
                                                          # le prétraitement local n'est pas persistant entre sessions)
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── CONFIG — mêmes données / mêmes seeds que YOLOv11n et Faster R-CNN (comparabilité) ────
IMG_SIZE      = 640
EPOCHS        = 150
PATIENCE      = 20        # aligné sur le run YOLO déjà effectué (comparabilité) — §4.4 protocole thèse prévoit 15, valeur portée à 20 ici pour matcher YOLO. À documenter comme valeur spécifique à ce volet article/FracAtlas.
WEIGHT_DECAY  = 0.0005    # = 5e-4, §4.4
AUG_DEGREES   = 8.0       # aligné sur ce qui a réellement tourné pour YOLO/Faster R-CNN (comparabilité)
AUG_TRANSLATE = 0.1
AUG_SCALE     = 0.15

CLAHE_CLIP    = 2.0
CLAHE_GRID    = (8, 8)
LABEL_TO_NAME = {0: 'Fracture'}

# Mêmes 5 seeds que YOLO et Faster R-CNN — étude de stabilité sur les mêmes graines
SEEDS = [42, 123, 2024, 7, 77]

with open(SPLITS_PATH) as f:
    splits = json.load(f)
with open(STATS_PATH) as f:
    clahe_stats = json.load(f)

print(f"Train : {len(splits['train'])} images | Val : {len(splits['val'])} images | Test : {len(splits['test'])} images")
print(f"CLAHE stats (calculées sur {clahe_stats['n_images']} images train) : mean={clahe_stats['mean'][0]:.4f}, std={clahe_stats['std'][0]:.4f}")
print(f"\nIMG_SIZE={IMG_SIZE} | EPOCHS={EPOCHS} | PATIENCE={PATIENCE} | SEEDS={SEEDS}")


Drive déjà monté
Train : 2857 images | Val : 613 images | Test : 613 images
CLAHE stats (calculées sur 2857 images train) : mean=0.2116, std=0.0552

IMG_SIZE=640 | EPOCHS=150 | PATIENCE=20 | SEEDS=[42, 123, 2024, 7, 77]


## 📦 Cellule 1bis — Extraction de l'archive FracAtlas

*(identique aux notebooks YOLO/Faster R-CNN)*

In [10]:
import zipfile

if os.path.isdir(FRACATLAS_ROOT) and len(os.listdir(FRACATLAS_ROOT)) > 0:
    print(f'✅ Déjà extrait dans cette session → {FRACATLAS_ROOT} ({len(os.listdir(FRACATLAS_ROOT))} éléments)')
else:
    assert os.path.exists(FRACATLAS_ZIP), f'Archive introuvable : {FRACATLAS_ZIP}'
    os.makedirs(FRACATLAS_ROOT, exist_ok=True)
    print(f'📦 Extraction de {FRACATLAS_ZIP} vers {FRACATLAS_ROOT}...')
    with zipfile.ZipFile(FRACATLAS_ZIP, 'r') as zf:
        zf.extractall(FRACATLAS_ROOT)
    print('✅ Extraction terminée.')

# Aperçu de l'arborescence extraite, pour confirmer/ajuster IMAGES_SUBDIRS et LABELS_DIR si besoin
print('\nContenu de premier niveau :', os.listdir(FRACATLAS_ROOT))
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    depth = root[len(FRACATLAS_ROOT):].count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} fichiers)' if files else '')


✅ Déjà extrait dans cette session → /content/FracAtlas_extracted (1 éléments)

Contenu de premier niveau : ['FracAtlas']
FracAtlas_extracted/ 
  FracAtlas/ (1 fichiers)
    Annotations/ 
    Utilities/ (2 fichiers)
    images/ 


## 🔎 Cellule 2 — Vérification de la structure sur Drive

*(identique aux notebooks précédents)*

In [11]:
# Vérifie la présence des dossiers images et labels, et échantillonne quelques fichiers
for sub in IMAGES_SUBDIRS:
    d = os.path.join(FRACATLAS_ROOT, 'images', sub)
    status = '✅' if os.path.isdir(d) else '⚠️  introuvable'
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{status}  images/{sub}  ({n} fichiers)  →  {d}')

status_lbl = '✅' if os.path.isdir(LABELS_DIR) else '⚠️  introuvable'
n_lbl = len(os.listdir(LABELS_DIR)) if os.path.isdir(LABELS_DIR) else 0
print(f'{status_lbl}  Annotations/YOLO  ({n_lbl} fichiers)  →  {LABELS_DIR}')

# Vérifie qu'un id du split train se retrouve bien quelque part (image + label)
sample_id = splits['train'][0]
stem = os.path.splitext(sample_id)[0]
found_img = None
for sub in IMAGES_SUBDIRS:
    p = os.path.join(FRACATLAS_ROOT, 'images', sub, sample_id)
    if os.path.exists(p):
        found_img = p
        break
found_lbl = os.path.join(LABELS_DIR, stem + '.txt')

print(f"\nTest sur '{sample_id}' :")
print(f'  image : {"✅ trouvée → " + found_img if found_img else "⚠️ NON trouvée dans " + str(IMAGES_SUBDIRS)}')
print(f'  label : {"✅ trouvé" if os.path.exists(found_lbl) else "⚠️ NON trouvé"} → {found_lbl}')
if os.path.exists(found_lbl):
    with open(found_lbl) as f:
        print(f'    contenu : {f.read().strip()}')
    print("    (format attendu : 'class_id x_center y_center width height', normalisé 0-1)")

print("\n⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.")


⚠️  introuvable  images/Fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Fractured
⚠️  introuvable  images/Non_fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Non_fractured
⚠️  introuvable  Annotations/YOLO  (0 fichiers)  →  /content/FracAtlas_extracted/Annotations/YOLO

Test sur 'IMG0003173.jpg' :
  image : ⚠️ NON trouvée dans ['Fractured', 'Non_fractured']
  label : ⚠️ NON trouvé → /content/FracAtlas_extracted/Annotations/YOLO/IMG0003173.txt

⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.


## 🔧 Cellule 3 — Prétraitement CLAHE + Letterbox + réécriture des labels YOLO

*(identique — mêmes images/labels produits par construction, CLAHE et letterbox étant déterministes)*

In [12]:
import cv2
import numpy as np
from tqdm.auto import tqdm

# ── Index robuste des images : balayage complet de FRACATLAS_ROOT ──
# Plutôt que de supposer une arborescence fixe (images/Fractured, images/Non_fractured...),
# on indexe UNE FOIS tous les fichiers image trouvés sous FRACATLAS_ROOT, par nom de fichier.
# Ça évite de dépendre d'une structure de dossiers qu'on ne connaît pas avec certitude.
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

print(f'🔎 Indexation des images sous {FRACATLAS_ROOT}...')
_image_index = {}
_n_walked = 0
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    for fname in files:
        if os.path.splitext(fname)[1].lower() in IMG_EXTENSIONS:
            _image_index[fname] = os.path.join(root, fname)
            _n_walked += 1
print(f'✅ {_n_walked} fichiers image indexés sous {FRACATLAS_ROOT}')

# Vérification immédiate sur un échantillon du split train (fail-fast si toujours 0 match)
_sample = splits['train'][:5]
_found = [i for i in _sample if i in _image_index]
print(f'   Échantillon train : {len(_found)}/{len(_sample)} trouvés → {_found}')
if len(_found) == 0:
    raise RuntimeError(
        f"Aucune image du split train n'a été retrouvée sous {FRACATLAS_ROOT}. "
        f"Vérifie FRACATLAS_ROOT en Cellule 1 (le dossier racine est-il le bon ?), "
        f"ou que le nom exact des fichiers (ex: '{_sample[0]}') correspond bien à ce qui est sur le Drive."
    )

def find_image_path(image_id):
    return _image_index.get(image_id, None)

# ── CORRECTIF : index robuste des LABELS, même logique que pour les images ──
# LABELS_DIR (Cellule 1) est un chemin statique qui suppose FRACATLAS_ROOT/Annotations/YOLO.
# Si le zip extrait une arborescence imbriquée (ex: FRACATLAS_ROOT/FracAtlas/Annotations/YOLO),
# ce chemin statique ne pointe vers rien, et l'ancien code traitait ça silencieusement comme
# "pas de label" pour TOUTES les images (bug identifié : 100% des images en background).
# On corrige en indexant tous les .txt trouvés sous FRACATLAS_ROOT par nom de fichier,
# restreint aux chemins contenant 'Annotations' pour éviter de capter un .txt parasite ailleurs.
print(f'🔎 Indexation des labels YOLO sous {FRACATLAS_ROOT}...')
_label_index = {}
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    if 'Annotations' not in root:
        continue
    for fname in files:
        if fname.endswith('.txt'):
            _label_index[fname] = os.path.join(root, fname)
print(f'✅ {len(_label_index)} fichiers .txt indexés sous Annotations/ (attendu : ~4083)')

def find_label_path(stem):
    return _label_index.get(stem + '.txt', None)

# Vérification fail-fast : on cherche spécifiquement des images FRACTURÉES du train
# (pas juste les 5 premières, qui sont probablement non-fracturées et auraient un label
# vide de toute façon — ça ne prouverait rien sur le bug qu'on corrige ici)
_fractured_sample = [
    i for i in splits['train']
    if find_image_path(i) and 'Fractured' in find_image_path(i) and 'Non_fractured' not in find_image_path(i)
][:5]
_labels_found = 0
for _fid in _fractured_sample:
    _stem = os.path.splitext(_fid)[0]
    _lp = find_label_path(_stem)
    if _lp and os.path.getsize(_lp) > 0:
        _labels_found += 1
print(f'   Échantillon train FRACTURÉ : {_labels_found}/{len(_fractured_sample)} ont un label non-vide '
      f'{_fractured_sample}')
if len(_fractured_sample) > 0 and _labels_found == 0:
    raise RuntimeError(
        "Aucun label non-vide trouvé pour des images fracturées du train. "
        "Le chemin Annotations/YOLO est peut-être introuvable dans l'arborescence extraite, "
        "ou le nommage des fichiers .txt ne correspond pas aux noms d'images. "
        "Vérifie la sortie de l'indexation ci-dessus (nombre de .txt trouvés)."
    )

def apply_clahe(img_gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    return clahe.apply(img_gray)

def letterbox_image(img, size=IMG_SIZE):
    """Redimensionne en conservant le ratio + padding centré. Retourne l'image + les paramètres
    de transformation (scale, pad_x, pad_y) nécessaires pour ajuster les boîtes."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_y, pad_x = (size - new_h) // 2, (size - new_w) // 2
    canvas = np.full((size, size), 114, dtype=np.uint8)  # gris neutre, convention YOLO
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y

def letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, size=IMG_SIZE):
    """Relit un fichier de labels YOLO (coords normalisées / image d'origine) et renvoie les
    lignes YOLO recalculées pour l'image letterboxée."""
    if label_path is None or not os.path.exists(label_path):
        return []  # image sans fracture annotée (ne devrait pas arriver si Fractured/ contient une box)
    new_lines = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, bw, bh = parts
            xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
            # normalisé -> pixels absolus (image d'origine)
            xc_px, yc_px = xc * orig_w, yc * orig_h
            bw_px, bh_px = bw * orig_w, bh * orig_h
            # applique le même scale + padding que l'image
            xc_new = xc_px * scale + pad_x
            yc_new = yc_px * scale + pad_y
            bw_new = bw_px * scale
            bh_new = bh_px * scale
            # renormalise par IMG_SIZE
            new_lines.append(f'{cls} {xc_new/size:.6f} {yc_new/size:.6f} {bw_new/size:.6f} {bh_new/size:.6f}')
    return new_lines

def preprocess_split(image_ids, split_name):
    img_out_dir = os.path.join(YOLO_DATA, 'images', split_name)
    lbl_out_dir = os.path.join(YOLO_DATA, 'labels', split_name)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    n_missing_img, n_no_label, n_with_label = 0, 0, 0
    for image_id in tqdm(image_ids, desc=f'Prétraitement {split_name}'):
        stem = os.path.splitext(image_id)[0]
        src_path = find_image_path(image_id)
        if src_path is None:
            n_missing_img += 1
            continue

        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = img.shape[:2]
        img_clahe = apply_clahe(img)
        img_lb, scale, pad_x, pad_y = letterbox_image(img_clahe, IMG_SIZE)
        img_rgb = cv2.cvtColor(img_lb, cv2.COLOR_GRAY2BGR)  # ultralytics attend du 3-canaux
        cv2.imwrite(os.path.join(img_out_dir, stem + '.jpg'), img_rgb)

        label_path = find_label_path(stem)  # CORRECTIF : index robuste au lieu du chemin statique
        new_lines = letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, IMG_SIZE)
        if label_path is None:
            n_no_label += 1
        elif new_lines:
            n_with_label += 1
        with open(os.path.join(lbl_out_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(new_lines))  # vide = image négative (Non_fractured), convention YOLO

    print(f'  ✅ {split_name} : {len(image_ids) - n_missing_img}/{len(image_ids)} images traitées '
          f'({n_missing_img} introuvables, {n_no_label} sans fichier .txt trouvé, '
          f'{n_with_label} avec au moins une bbox)')

for split_name, ids in [('train', splits['train']), ('val', splits['val']), ('test', splits['test'])]:
    preprocess_split(ids, split_name)


🔎 Indexation des images sous /content/FracAtlas_extracted...
✅ 4083 fichiers image indexés sous /content/FracAtlas_extracted
   Échantillon train : 5/5 trouvés → ['IMG0003173.jpg', 'IMG0000063.jpg', 'IMG0002055.jpg', 'IMG0002154.jpg', 'IMG0001769.jpg']
🔎 Indexation des labels YOLO sous /content/FracAtlas_extracted...
✅ 4084 fichiers .txt indexés sous Annotations/ (attendu : ~4083)
   Échantillon train FRACTURÉ : 5/5 ont un label non-vide ['IMG0002607.jpg', 'IMG0002483.jpg', 'IMG0002577.jpg', 'IMG0002127.jpg', 'IMG0001494.jpg']


Prétraitement train:   0%|          | 0/2857 [00:00<?, ?it/s]

  ✅ train : 2857/2857 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 513 avec au moins une bbox)


Prétraitement val:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ val : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 96 avec au moins une bbox)


Prétraitement test:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ test : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 108 avec au moins une bbox)


## ⚖️ Cellule 3bis — Suréchantillonnage des images fracturées (train uniquement)

*(identique — même TARGET_POSITIVE_RATIO=0.37)*

In [13]:
import shutil

TARGET_POSITIVE_RATIO = 0.37  # ~35-40%, choix validé (cf. cellule markdown ci-dessus)

train_img_dir = os.path.join(YOLO_DATA, 'images', 'train')
train_lbl_dir = os.path.join(YOLO_DATA, 'labels', 'train')

# Identifie les images positives (label non vide) vs négatives dans le train prétraité
all_stems = [os.path.splitext(f)[0] for f in os.listdir(train_img_dir)]
pos_stems, neg_stems = [], []
for stem in all_stems:
    lbl_path = os.path.join(train_lbl_dir, stem + '.txt')
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        pos_stems.append(stem)
    else:
        neg_stems.append(stem)

n_pos, n_neg = len(pos_stems), len(neg_stems)
current_ratio = n_pos / (n_pos + n_neg)
print(f'Avant suréchantillonnage : {n_pos} positifs / {n_pos + n_neg} total ({current_ratio:.1%})')

# k = nombre total de copies par image positive (originale incluse) pour atteindre TARGET_POSITIVE_RATIO
# dérivé de : n_pos*k / (n_pos*k + n_neg) = TARGET_POSITIVE_RATIO
k = (TARGET_POSITIVE_RATIO * n_neg) / (n_pos * (1 - TARGET_POSITIVE_RATIO))
k_extra = max(0, round(k) - 1)  # nombre de duplicatas à ajouter par image positive
print(f'Facteur de duplication calculé : k={k:.2f} → {k_extra} copie(s) supplémentaire(s) par image positive')

n_created = 0
for stem in pos_stems:
    src_img = os.path.join(train_img_dir, stem + '.jpg')
    src_lbl = os.path.join(train_lbl_dir, stem + '.txt')
    for dup_i in range(1, k_extra + 1):
        dst_img = os.path.join(train_img_dir, f'{stem}_dup{dup_i}.jpg')
        dst_lbl = os.path.join(train_lbl_dir, f'{stem}_dup{dup_i}.txt')
        if not os.path.exists(dst_img):
            shutil.copyfile(src_img, dst_img)
            shutil.copyfile(src_lbl, dst_lbl)
            n_created += 1

n_pos_after = n_pos * (1 + k_extra)
n_total_after = n_pos_after + n_neg
print(f'✅ {n_created} paires image/label créées par duplication (train uniquement)')
print(f'Après suréchantillonnage : {n_pos_after} positifs / {n_total_after} total ({n_pos_after / n_total_after:.1%})')
print('⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.')

Avant suréchantillonnage : 1539 positifs / 3883 total (39.6%)
Facteur de duplication calculé : k=0.89 → 0 copie(s) supplémentaire(s) par image positive
✅ 0 paires image/label créées par duplication (train uniquement)
Après suréchantillonnage : 1539 positifs / 3883 total (39.6%)
⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.


## 📦 Cellule 4 — data.yaml + installation ultralytics

In [14]:
import subprocess, sys
# Même version épinglée que YOLO/Faster R-CNN : ultralytics==8.3.40 supporte nativement RT-DETR
# (classe RTDETR) — garantit aussi qu'on charge bien rtdetr-l.pt et pas une variante ultérieure.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.40'], check=True)

data_yaml_path = os.path.join(YOLO_DATA, 'data.yaml')
data_yaml_content = f"""path: {YOLO_DATA}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']
"""
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print('✅ ultralytics installé, data.yaml écrit :')
print(data_yaml_content)


✅ ultralytics installé, data.yaml écrit :
path: /content/rtdetr_fracatlas_dataset
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']



## 🚀 Cellule 5 — Entraînement multi-seed RT-DETR-l

Checkpoints repris depuis le disque si déjà entraînés (même convention que YOLO/Faster R-CNN).

In [15]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.40'], check=True)

data_yaml_path = os.path.join(YOLO_DATA, 'data.yaml')
data_yaml_content = f"""path: {YOLO_DATA}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']
"""
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print('✅ ultralytics installé, data.yaml écrit :')
print(data_yaml_content)

import os, yaml, pandas as pd

print(f"{'seed':<6}{'lr0 réel':<10}{'nbs':<6}{'amp':<7}{'epochs':<8}{'max mAP50 val':<15}{'NaN dans les pertes'}")
for seed in SEEDS:
    d = os.path.join(SAVE_DIR, f'seed_{seed}')
    args_path, csv_path = os.path.join(d, 'args.yaml'), os.path.join(d, 'results.csv')
    if not os.path.exists(args_path):
        print(f'{seed:<6}(dossier absent ou entraînement jamais démarré)')
        continue
    a = yaml.safe_load(open(args_path))
    n_ep, best_map, has_nan = '-', '-', '-'
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        n_ep = len(df)
        if 'metrics/mAP50(B)' in df.columns:
            best_map = f"{df['metrics/mAP50(B)'].max():.4f}"
        loss_cols = [c for c in df.columns if c.startswith('train/') or c.startswith('val/')]
        has_nan = 'OUI' if df[loss_cols].isna().any().any() else 'non'
    print(f"{seed:<6}{str(a.get('lr0')):<10}{str(a.get('nbs')):<6}{str(a.get('amp')):<7}{str(n_ep):<8}{best_map:<15}{has_nan}")
print("\nLecture : lr0 attendu = 0.0001. Une valeur de 0.01 = défaut ultralytics (lr0 non transmis) → seed à refaire.")

from ultralytics import RTDETR
import os, shutil, torch, gc, pandas as pd

LR0_RTDETR = globals().get('LR0_RTDETR', 1e-4)
print(f'LR0_RTDETR utilisé pour tous les entraînements ci-dessous : {LR0_RTDETR}')
assert LR0_RTDETR == 1e-4, f'LR0_RTDETR={LR0_RTDETR} : valeur inattendue (attendu 1e-4)'

trained_weights = {}  # seed -> chemin vers best.pt


def read_results(seed_dir):
    csv_path = os.path.join(seed_dir, 'results.csv')
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df


def has_diverged(seed_dir):
    """True si une perte (train/ ou val/) contient un NaN : le run a divergé."""
    df = read_results(seed_dir)
    if df is None:
        return False
    loss_cols = [c for c in df.columns if c.startswith('train/') or c.startswith('val/')]
    return bool(df[loss_cols].isna().any().any())


def training_is_complete(seed_dir, max_epochs, patience, metric_col='metrics/mAP50(B)'):
    """Toutes les epochs faites, ou arrêt anticipé réellement déclenché (patience atteinte)."""
    df = read_results(seed_dir)
    if df is None:
        return False
    n_epochs_done = len(df)
    if n_epochs_done >= max_epochs:
        return True
    if metric_col not in df.columns:
        return False
    best_epoch = df[metric_col].idxmax()
    return (n_epochs_done - 1 - best_epoch) >= patience


def run_training(seed, seed_dir, resume, model_src):
    model = RTDETR(model_src)
    model.train(
        data=data_yaml_path,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=4,
        nbs=16,
        seed=seed,
        project=SAVE_DIR,
        name=f'seed_{seed}',
        exist_ok=True,
        resume=resume,
        optimizer='AdamW',
        lr0=LR0_RTDETR,
        lrf=0.01,
        weight_decay=WEIGHT_DECAY,
        cos_lr=True,
        warmup_epochs=3.0,
        amp=True,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        degrees=AUG_DEGREES, translate=AUG_TRANSLATE, scale=AUG_SCALE,
        shear=0.0, perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=0.0, mixup=0.0,
    )


for seed in SEEDS:
    seed_dir = os.path.join(SAVE_DIR, f'seed_{seed}')
    weights_dir = os.path.join(seed_dir, 'weights')
    best_ckpt = os.path.join(weights_dir, 'best.pt')
    last_ckpt = os.path.join(weights_dir, 'last.pt')

    if has_diverged(seed_dir):
        raise RuntimeError(
            f'Seed {seed} : des NaN sont présents dans results.csv (entraînement divergé). '
            f'Renomme le dossier {seed_dir} (ex. seed_{seed}_nan) pour le garder comme trace, '
            f'puis relance la cellule : le seed sera réentraîné proprement.'
        )

    if os.path.exists(best_ckpt) and training_is_complete(seed_dir, EPOCHS, PATIENCE):
        print(f'✅ Seed {seed} : entraînement complet, chargement → {best_ckpt}')
        trained_weights[seed] = best_ckpt
        continue

    resume = os.path.exists(last_ckpt)
    if resume:
        print(f'\n🔄 Seed {seed} — entraînement incomplet détecté, reprise depuis last.pt...')
        try:
            run_training(seed, seed_dir, True, last_ckpt)
        except AssertionError as e:
            print(f'⚠️ {e}\n   → repli : nouvel entraînement complet depuis les poids COCO.')
            run_training(seed, seed_dir, False, 'rtdetr-l.pt')
    else:
        print(f'\n🚀 Seed {seed} — entraînement RT-DETR-l (640px)...')
        run_training(seed, seed_dir, False, 'rtdetr-l.pt')

    if has_diverged(seed_dir):
        raise RuntimeError(
            f'Seed {seed} : NaN détectés pendant cet entraînement. Seed non validé. '
            f'Renomme {seed_dir} avant toute nouvelle tentative.'
        )

    trained_weights[seed] = best_ckpt
    print(f'✅ Seed {seed} — terminé, meilleur checkpoint → {best_ckpt}')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n🏁 Entraînement terminé pour les {len(SEEDS)} seeds.')

✅ ultralytics installé, data.yaml écrit :
path: /content/rtdetr_fracatlas_dataset
train: images/train
val: images/val
test: images/test
nc: 1
names: ['Fracture']

seed  lr0 réel  nbs   amp    epochs  max mAP50 val  NaN dans les pertes
42    0.0001    64    True   103     0.5853         non
123   0.0001    64    True   138     0.5918         non
2024  0.0001    64    True   105     0.5708         non
7     0.0001    64    True   68      0.5397         non
77    0.0001    64    True   75      0.5615         non

Lecture : lr0 attendu = 0.0001. Une valeur de 0.01 = défaut ultralytics (lr0 non transmis) → seed à refaire.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
LR0_RTDETR utilisé pour tous les entraînements ci-dessous : 0.00

## 📊 Cellule 6 — Évaluation multi-seed (AUC, sensibilité, spécificité, mAP50)

Identique à la logique du notebook YOLO (mAP@IoU=0.5 via `model.val()`, score image-level via `model.predict()`, AUC/sensibilité/spécificité au seuil de Youden, IC95% bootstrap n=1000).

In [16]:
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np

results_by_seed = {}

for seed in SEEDS:
    metrics_path = os.path.join(SAVE_DIR, f'metriques_seed{seed}.json')

    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            cached = json.load(f)
        if 'mAP50' in cached.get('Fracture', {}):
            results_by_seed[seed] = cached
            print(f'✅ Seed {seed} : métriques déjà calculées, chargées depuis le disque.')
            continue

    print(f'\n📊 Seed {seed} — évaluation...')
    model = RTDETR(trained_weights[seed])

    # mAP50 (IoU=0.5 uniquement, cohérence avec le protocole thoracique — cf. correctif YOLO/Faster R-CNN)
    val_results = model.val(data=data_yaml_path, split='test', imgsz=IMG_SIZE, iou=0.5, conf=0.001)
    map50 = float(val_results.box.map50)

    # Score image-level (max de confiance des boîtes prédites, 0 si aucune détection)
    test_dir = os.path.join(YOLO_DATA, 'images', 'test')
    labels_dir = os.path.join(YOLO_DATA, 'labels', 'test')
    y_true, y_score = [], []
    for fname in sorted(os.listdir(test_dir)):
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(labels_dir, stem + '.txt')
        has_fracture = os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0
        pred = model.predict(os.path.join(test_dir, fname), imgsz=IMG_SIZE, conf=0.001, verbose=False)[0]
        score = float(pred.boxes.conf.max()) if len(pred.boxes) > 0 else 0.0
        y_true.append(int(has_fracture))
        y_score.append(score)

    y_true, y_score = np.array(y_true), np.array(y_score)
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    youden_idx = np.argmax(tpr - fpr)
    best_thr = float(thresholds[youden_idx])
    sensib = float(tpr[youden_idx])
    specif = float(1 - fpr[youden_idx])

    # Bootstrap IC95% de l'AUC — identique à YOLO/Faster R-CNN (n=1000)
    rng = np.random.default_rng(seed)
    boot_aucs = []
    n = len(y_true)
    for _ in range(1000):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])

    result = {
        'Fracture': {
            'AUC': round(float(auc), 4),
            'IC95': [round(float(ci_low), 4), round(float(ci_high), 4)],
            'Sensibilite': round(sensib, 4),
            'Specificite': round(specif, 4),
            'Seuil_Youden': round(best_thr, 4),
            'mAP50': round(map50, 4),
        },
        'global': {
            'model': 'RTDETR-l',
            'mAP50': round(map50, 4),
            'seed': seed,
            'img_size': IMG_SIZE,
            'preprocessing': f'CLAHE(clip={CLAHE_CLIP})+Letterbox{IMG_SIZE}',
        },
    }
    with open(metrics_path, 'w') as f:
        json.dump(result, f, indent=2)
    results_by_seed[seed] = result
    print(f'✅ Seed {seed} — AUC={auc:.4f} | mAP50={map50:.4f}')

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



📊 Seed 42 — évaluation...
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 506 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 109MB/s]
val: Scanning /content/rtdetr_fracatlas_dataset/labels/test... 613 images, 505 backgrounds, 0 corrupt: 100%|██████████| 613/613 [00:00<00:00, 2899.88it/s]

val: New cache created: /content/rtdetr_fracatlas_dataset/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:22<00:00,  1.75it/s]


                   all        613        134      0.719      0.433      0.449      0.173
Speed: 0.5ms preprocess, 32.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/detect/val
✅ Seed 42 — AUC=0.8670 | mAP50=0.4491

📊 Seed 123 — évaluation...
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 502 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs


val: Scanning /content/rtdetr_fracatlas_dataset/labels/test.cache... 613 images, 505 backgrounds, 0 corrupt: 100%|██████████| 613/613 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:23<00:00,  1.67it/s]


... (3 outputs left out)

val: Scanning /content/rtdetr_fracatlas_dataset/labels/test.cache... 613 images, 505 backgrounds, 0 corrupt: 100%|██████████| 613/613 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:23<00:00,  1.66it/s]


                   all        613        134      0.659      0.396      0.356      0.134
Speed: 0.9ms preprocess, 35.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/detect/val4
✅ Seed 7 — AUC=0.8731 | mAP50=0.3562

📊 Seed 77 — évaluation...
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 506 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs


val: Scanning /content/rtdetr_fracatlas_dataset/labels/test.cache... 613 images, 505 backgrounds, 0 corrupt: 100%|██████████| 613/613 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:23<00:00,  1.67it/s]


                   all        613        134      0.736      0.438      0.417      0.153
Speed: 0.8ms preprocess, 35.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/detect/val5
✅ Seed 77 — AUC=0.8564 | mAP50=0.4169


## 🏁 Cellule 7 — Agrégation sur les 5 seeds

In [17]:
aucs = [results_by_seed[s]['Fracture']['AUC'] for s in SEEDS]
maps = [results_by_seed[s]['Fracture']['mAP50'] for s in SEEDS]

agg = {
    'Fracture': {
        'AUC_mean': round(float(np.mean(aucs)), 4),
        'AUC_std': round(float(np.std(aucs)), 4),
        'mAP50_mean': round(float(np.mean(maps)), 4),
        'mAP50_std': round(float(np.std(maps)), 4),
        'n_seeds': len(SEEDS),
    },
    'global': {'model': 'RTDETR-l', 'seeds': SEEDS},
}
with open(os.path.join(SAVE_DIR, 'resultats_agreges_5seeds.json'), 'w') as f:
    json.dump(agg, f, indent=2)

print(f"AUC   = {agg['Fracture']['AUC_mean']:.4f} ± {agg['Fracture']['AUC_std']:.4f}")
print(f"mAP50 = {agg['Fracture']['mAP50_mean']:.4f} ± {agg['Fracture']['mAP50_std']:.4f}")


AUC   = 0.8699 ± 0.0089
mAP50 = 0.4019 ± 0.0307


## 📐 Cellule 8 — Précision, rappel et F1 (boîtes) — RT-DETR-l, 5 graines

À lancer après la Cellule 6 (réutilise `YOLO_DATA`, `SAVE_DIR`, `data_yaml_path`, `IMG_SIZE`, `SEEDS`). Poids : `SAVE_DIR/seed_X/weights/best.pt`.

- **Vérification préalable** : la cellule lit `args.yaml` de chaque graine, affiche nbs / batch / optimizer / lr0 et s'arrête si `nbs` ≠ `EXPECTED_NBS` (= 64).
- Une prédiction est un vrai positif si son IoU avec une boîte annotée est ≥ 0,5 (appariement glouton par confiance décroissante, une boîte annotée ne peut être appariée qu'une fois). RT-DETR n'a pas de NMS : les doublons de boîtes comptent comme faux positifs.
- Le seuil de confiance maximise le **F1 de validation** (boîtes), sur la grille 0,05–0,95 par pas de 0,05 (identique à la section 2.5), puis il est appliqué au test.
- Les images normales comptent : toute boîte prédite dessus est un faux positif. On rapporte aussi les FP par image normale et la spécificité image à ce seuil.
- Le mAP@50 est recalculé et comparé à celui de `metriques_seed{seed}.json`.
- Moyenne ± écart-type avec **ddof = 1** (n = 5 graines). Sauvegarde dans `metriques_detection_seed{seed}.json` et `resultats_detection_agreges.json`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 8 — Précision / Rappel / F1 au niveau des boîtes (RT-DETR-l, 5 graines)
# Seuil de confiance choisi sur la VALIDATION (maximise le F1), appliqué au TEST.
# Poids utilisés : SAVE_DIR/seed_X/weights/best.pt — entraînements réalisés avec nbs=64 (args.yaml fait foi)
# ═══════════════════════════════════════════════════════════════════════════
import os, json, gc
import numpy as np
import torch
from ultralytics import RTDETR
import yaml

MODEL_DIR = SAVE_DIR            # dossier RTDETR (seed_X/weights/best.pt)
RUN_SEEDS = list(SEEDS)         # mêmes 5 graines que la Cellule 1
EXPECTED_NBS = 64               # nbs réellement utilisé pour les résultats retenus (vérifié dans args.yaml)
IOU_MATCH = 0.5      # une prédiction est un vrai positif si IoU >= 0.5 avec une boîte annotée
# RT-DETR n'utilise pas de NMS (sortie directe des requêtes du décodeur, 300 max par image)
CONF_MIN  = 0.001    # on garde tout, le seuil est appliqué ensuite
THR_GRID  = np.round(np.arange(0.05, 0.951, 0.05), 2)   # même grille que la section 2.5 de l'article (0,05 à 0,95, pas de 0,05)
# égalité de F1 → np.argmax retient le seuil le plus faible, comme décrit en 2.5


def _load_gt(lbl_path, size):
    """Boîtes annotées (xyxy, pixels de l'image letterboxée). Vide si image normale."""
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(lbl_path, ndmin=2)
    xc, yc, w, h = a[:, 1] * size, a[:, 2] * size, a[:, 3] * size, a[:, 4] * size
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], axis=1)


def _iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


CHUNK = 8    # images par lot (RT-DETR-l est lourd) : évite l'OOM


def _collect(model, split):
    """Prédictions (toutes confiances >= CONF_MIN) + vérité terrain, image par image."""
    img_dir = os.path.join(YOLO_DATA, 'images', split)
    lbl_dir = os.path.join(YOLO_DATA, 'labels', split)
    fnames = sorted(os.listdir(img_dir))
    data = []
    for i in range(0, len(fnames), CHUNK):
        chunk = fnames[i:i + CHUNK]
        results = model.predict([os.path.join(img_dir, f) for f in chunk], imgsz=IMG_SIZE,
                                conf=CONF_MIN, batch=CHUNK, verbose=False)
        for fname, r in zip(chunk, results):
            boxes = r.boxes.xyxy.cpu().numpy() if len(r.boxes) else np.zeros((0, 4))
            scores = r.boxes.conf.cpu().numpy() if len(r.boxes) else np.zeros(0)
            order = np.argsort(-scores)
            boxes, scores = boxes[order], scores[order]
            gt = _load_gt(os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt'), IMG_SIZE)
            data.append({'scores': scores, 'n_gt': len(gt), 'iou': _iou_matrix(boxes, gt)})
        del results
    return data


def _counts(data, thr):
    """TP/FP/FN au niveau des boîtes, appariement glouton par confiance décroissante."""
    tp = fp = fn = 0
    n_normal = fp_normal = normal_flagged = 0
    for d in data:
        n_pred = int((d['scores'] >= thr).sum())     # scores triés : les n_pred premiers
        n_gt = d['n_gt']
        taken = np.zeros(n_gt, dtype=bool)
        tp_i = 0
        for i in range(n_pred):
            if n_gt == 0:
                break
            ious = np.where(taken, -1.0, d['iou'][i])
            j = int(np.argmax(ious))
            if ious[j] >= IOU_MATCH:
                taken[j] = True
                tp_i += 1
        tp += tp_i; fp += n_pred - tp_i; fn += n_gt - tp_i
        if n_gt == 0:
            n_normal += 1; fp_normal += n_pred; normal_flagged += int(n_pred > 0)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'Precision': prec, 'Rappel': rec, 'F1': f1,
            'FP_par_image_normale': fp_normal / n_normal if n_normal else float('nan'),
            'Specificite_image': 1 - normal_flagged / n_normal if n_normal else float('nan')}


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Vérification avant de lancer : hyperparamètres réellement utilisés (args.yaml de chaque graine) ──
print(f"{'seed':<6}{'nbs':<6}{'batch':<7}{'optimizer':<11}{'lr0':<9}{'imgsz':<7}{'epochs'}")
_args = {}
for seed in RUN_SEEDS:
    p = os.path.join(MODEL_DIR, f'seed_{seed}', 'args.yaml')
    assert os.path.exists(p), f'args.yaml introuvable pour la graine {seed} : {p}'
    a = yaml.safe_load(open(p))
    _args[seed] = a
    print(f"{seed:<6}{str(a.get('nbs')):<6}{str(a.get('batch')):<7}{str(a.get('optimizer')):<11}"
          f"{str(a.get('lr0')):<9}{str(a.get('imgsz')):<7}{a.get('epochs')}")
_nbs = {a.get('nbs') for a in _args.values()}
assert len(_nbs) == 1, f'nbs différent selon les graines : {_nbs} → vérifier les entraînements.'
assert _nbs == {EXPECTED_NBS}, (f"args.yaml indique nbs={_nbs.pop()} mais EXPECTED_NBS={EXPECTED_NBS}. "
                                f"Corriger EXPECTED_NBS si c'est args.yaml qui fait foi, puis relancer.")
print(f'✅ nbs={EXPECTED_NBS} pour les {len(RUN_SEEDS)} graines.\n')

detect_by_seed = {}
for seed in RUN_SEEDS:
    out_path = os.path.join(MODEL_DIR, f'metriques_detection_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            detect_by_seed[seed] = json.load(f)
        print(f'✅ Seed {seed} : métriques de détection déjà calculées, chargées.')
        continue

    print(f'\n📐 Seed {seed} — précision / rappel / F1...')
    weights = os.path.join(MODEL_DIR, f'seed_{seed}', 'weights', 'best.pt')
    assert os.path.exists(weights), f'Poids introuvables (seed {seed}) : {weights}'
    model = RTDETR(weights)

    # 1) seuil de confiance qui maximise le F1 (boîtes) sur la validation
    val_data = _collect(model, 'val')
    f1_val = np.array([_counts(val_data, t)['F1'] for t in THR_GRID])
    thr = float(THR_GRID[int(np.argmax(f1_val))])

    # 2) métriques sur le test à ce seuil
    test_data = _collect(model, 'test')
    m = _counts(test_data, thr)

    # 3) mAP@50 et mAP@50-95 (même appel que la Cellule 6, iou=0.5 = NMS)
    vr = model.val(data=data_yaml_path, split='test', imgsz=IMG_SIZE,
                   iou=0.5, conf=0.001, batch=8, verbose=False, plots=False)
    map50, map5095 = float(vr.box.map50), float(vr.box.map)

    # Contrôle de cohérence : le mAP@50 recalculé doit retrouver celui déjà enregistré
    prev = os.path.join(MODEL_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(prev):
        with open(prev) as f:
            prev_map50 = json.load(f)['Fracture']['mAP50']
        if abs(prev_map50 - map50) > 0.002:
            print(f'   ⚠️ mAP50 recalculé ({map50:.4f}) ≠ mAP50 enregistré ({prev_map50:.4f}) : '
                  f'vérifier que ce sont bien les mêmes poids.')
    if map50 < 0.1:
        print("   ⚠️ mAP50 anormalement bas : checkpoint probablement issu d'un entraînement interrompu.")

    res = {'seuil_F1_val': thr, 'F1_val_max': round(float(f1_val.max()), 4),
           'mAP50': round(map50, 4), 'mAP50_95': round(map5095, 4),
           'nbs': _args[seed].get('nbs'), 'batch': _args[seed].get('batch')}
    res.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
    with open(out_path, 'w') as f:
        json.dump(res, f, indent=2)
    detect_by_seed[seed] = res
    print(f"   seuil={thr:.3f} | P={res['Precision']:.3f} R={res['Rappel']:.3f} F1={res['F1']:.3f} "
          f"| FP/img normale={res['FP_par_image_normale']:.3f} | mAP50-95={map5095:.3f}")

    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Agrégation : moyenne ± écart-type (ddof=1, correction de Bessel, n = nb de graines) ──
keys = ['mAP50', 'Precision', 'Rappel', 'F1', 'mAP50_95', 'TP', 'FP', 'FN', 'FP_par_image_normale', 'Specificite_image', 'seuil_F1_val']
agg_det = {'n_seeds': len(RUN_SEEDS), 'poids': 'RTDETR-l (nbs=%d)' % EXPECTED_NBS, 'ddof': 1}
print('\nRT-DETR-l — détection (boîtes, IoU >= 0.5), moyenne ± é.-t. (ddof=1) sur', len(RUN_SEEDS), 'graines')
for k in keys:
    v = np.array([detect_by_seed[s][k] for s in RUN_SEEDS], dtype=float)
    agg_det[k] = {'mean': round(float(v.mean()), 4), 'std': round(float(v.std(ddof=1)), 4)}
    print(f"  {k:<22s} {v.mean():.3f} ± {v.std(ddof=1):.3f}")
with open(os.path.join(MODEL_DIR, 'resultats_detection_agreges.json'), 'w') as f:
    json.dump(agg_det, f, indent=2)
